# OCR DETECTION (NUXIA PROJECT):

## 1. Import libraries and dependancies

In [ ]:
#pip install paddleocr
#pip install paddlepaddle
#pip install pytesseract # OTRO OCR

In [32]:
from paddleocr import PaddleOCR
from pathlib import Path
import cv2
import numpy as np
import shutil
#import pytesseract
import json
import re
from datetime import datetime


### Data Structure:

/main 


//pruebas,//pruebas,//pruebas

//data_train (entrenar el classifier)
//data_all (divididas en troqueladas y no con sus previos análisis)

## 2. Classification model for die cut images

### 2.1. Pipeline to obtain the data to train the classifier

In [ ]:
# DO NOT RUN

path = #

In [ ]:
# DO NOT RUN

CARPETA_ORIGEN = path / "pruebas14082026"
CARPETA_DESTINO = path / "data_train"
EXTENSIONES = {".png", ".jpg", ".jpeg"}

CARPETA_DESTINO.mkdir(exist_ok=True)

copiadas = 0
sin_original = []

for subcarpeta in sorted(p for p in CARPETA_ORIGEN.iterdir() if p.is_dir()):
    candidatas = [
        f for f in subcarpeta.iterdir()
        if f.stem.lower() == "original" and f.suffix.lower() in EXTENSIONES
    ]

    if not candidatas:
        sin_original.append(subcarpeta.name)
        continue

    origen = candidatas[0]
    destino_dir = CARPETA_DESTINO / subcarpeta.name
    destino_dir.mkdir(exist_ok=True)

    # copia la imagen (el original queda intacto)
    shutil.copy2(origen, destino_dir / f"original{origen.suffix.lower()}")

    # crea el txt vacío para etiquetar, sin pisar uno que ya hayas rellenado
    etiqueta = destino_dir / "etiqueta.txt"
    if not etiqueta.exists():
        etiqueta.touch()

    copiadas += 1


### 2.2 Train the classifier

In [ ]:
#

### Once we have the final images let's work with them

## 3. Image Preprocessing

In [28]:
paddle_ocr = PaddleOCR(
    lang="es",
    use_angle_cls=True,
    enable_mkldnn=False,                          # idioma (o "en", "ch", etc.)
    use_doc_orientation_classify=False, # desactiva módulos extra para ir más rápido
    use_doc_unwarping=False
)

C:\Users\sampe\AppData\Local\Temp\ipykernel_34952\401327702.py:1: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  paddle_ocr = PaddleOCR(
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `C:\Users\sampe\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sampe\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\sampe\.paddlex\official_models\PP-OCRv6_medium_rec`.


In [ ]:
# Imágenes para utilizar (Copiar y pegar) en path_img = Path("pruebas14082026/1b4df212")

# 1a0b72b0 (Diferente formato) - Tiene que extraer todo
# 1a3b95bb (Troquelada) - No se extrae nada
# 1aaa2407 (Diferente formato) - Tiene que extraer todo
# 1b4df212 (Diferente formato) - Tiene que extraer todo
# 2b733ecd (Diferente formato) - Tiene que extraer todo
# 2c9787d9 (Diferente formato) - Tiene que extraer todo
# 4d95d836 (Diferente formato) - Tiene que extraer todo

In [ ]:
# Problemas:

# 1a0b72b0 - Junta dos líneas de texto
# 1a3b95bb
# 1aaa2407 - Lote: en Lote1 y vcto: en uct01
# Los dos puntos los convierte en un 1
# 1b4df212 - Está rotada y no entiende bien el número de lote
# 2b733ecd - Junta dos líneas de texto
# 2c9787d9 - "Lote N° :"
# 4d95d836 - LOTE:, y FP Y FE

# ¿Cómo lidiar con rotadas? use_angle_cls=True
# Junta dos cadenas de texto (arriba y abajo en una)
# Convierte dos puntos en "1"

In [40]:
path_img = Path("pruebas14082026/2c9787d9") #Cambiar si trabajamos con otra carpeta

In [30]:
img = cv2.imread(str(path_img / "original.png"))

In [31]:
resultado = paddle_ocr.predict(img)

for res in resultado:
    #res.print()                      # muestra todo en consola
    res.save_to_img(str(path_img / "ocr_no_pre.png"))        # imagen con las cajas dibujadas
    res.save_to_json(str(path_img / "report.json"))     # JSON con textos y coordenadas

Extraer el texto

In [50]:
ruta_json = path_img / "report.json"

with open(ruta_json, "r", encoding="utf-8") as f:
    report = json.load(f)

rec_texts = report["rec_texts"]

# No modificar ni normalizar el texto original
texto = [str(text) for text in rec_texts]

# Unir los textos para buscar etiquetas o fechas
texto = "\n".join(texto)

print(texto)

Lote N° : FD250359
Fecha de Fab.: 01/2025
Fecha de Exp.: 12/2026


Función Maestra:

In [53]:
def funcion_maestra(texto):

    resultado = {
        "texto_original": texto,
        "lote": None,
        "fecha_produccion": None,
        "fecha_caducidad": None,
    }

    # Buscar lote: LOT, LOTE, Lote N° :, LOT NO, BATCH, etc.
    patron_lote = re.compile(
    r"\b(?:LOTE|LOT|BATCH)\b"        # Etiqueta
    r"(?:\s*(?:N\s*[°ºª.]|NO\.?|NUM(?:ERO)?))?"  # N°, Nº, No, Número
    r"\s*[:：#=\-]?\s*"              # Separadores opcionales
    r"([A-Z0-9][A-Z0-9./_-]*)",      # Valor del lote
    re.IGNORECASE
    )

    coincidencia_lote = patron_lote.search(texto)

    if coincidencia_lote:
        resultado["lote"] = coincidencia_lote.group(1).strip()


    # 3. Buscar fechas en formatos habituales
    patron_fecha = re.compile(
        r"(?<!\d)("
        r"\d{4}[-/.]\d{1,2}[-/.]\d{1,2}"  # YYYY-MM-DD
        r"|"
        r"\d{1,2}[-/.]\d{1,2}[-/.]\d{4}"  # DD/MM/YYYY
        r"|"
        r"\d{1,2}[-/.]\d{4}"              # MM/YYYY
        r")(?!\d)"
    )

    formatos = [
        "%Y-%m-%d", "%Y/%m/%d", "%Y.%m.%d",
        "%d/%m/%Y", "%d-%m-%Y", "%d.%m.%Y",
        "%m/%Y", "%m-%Y", "%m.%Y",
    ]

    fechas_encontradas = []
    fechas_vistas = set()

    for match in patron_fecha.finditer(texto):
        fecha_texto = match.group(1) 

        for formato in formatos:
            try:
                fecha = datetime.strptime(fecha_texto, formato)

                # Para MM/YYYY, se utiliza el primer día del mes
                # solo para poder comparar cronológicamente.
                if formato in ("%m/%Y", "%m-%Y", "%m.%Y"):
                    fecha = fecha.replace(day=1)

                if fecha_texto not in fechas_vistas:
                    fechas_encontradas.append(
                        (fecha, fecha_texto)
                    )
                    fechas_vistas.add(fecha_texto)

                break

            except ValueError:
                continue

    # 4. Clasificar las fechas por orden cronológico
    fechas_encontradas.sort(key=lambda x: x[0])

    if len(fechas_encontradas) == 1:
        resultado["fecha_caducidad"] = (
            fechas_encontradas[0][1]
        )

    elif len(fechas_encontradas) >= 2:
        resultado["fecha_produccion"] = (
            fechas_encontradas[0][1]
        )
        resultado["fecha_caducidad"] = (
            fechas_encontradas[-1][1]
        )

    return resultado

Ejemplo:

In [54]:
# Ejemplo de uso
datos = funcion_maestra(texto)

print(json.dumps(datos, indent=4, ensure_ascii=False))

{
    "texto_original": "Lote N° : FD250359\nFecha de Fab.: 01/2025\nFecha de Exp.: 12/2026",
    "lote": "FD250359",
    "fecha_produccion": "01/2025",
    "fecha_caducidad": "12/2026"
}


### Ya tenemos nuestra cadena de texto limpia, lo guardaremos en un .txt para posterior uso

### Let's try with different filters

In [ ]:
# GRIS

gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
imagen_limpia = clahe.apply(gris)
cv2.imwrite("salida/gris.png", imagen_limpia)

imagen_3c = cv2.cvtColor(imagen_limpia, cv2.COLOR_GRAY2BGR) # Necesitas tres canales ya que Paddle OCR lo requiere

for res in paddle_ocr.predict(imagen_3c):
    res.print()
    res.save_to_img("salida")        # imagen con las cajas dibujadas
    res.save_to_json("salida") 

True

In [ ]:
# CLAHE

lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)
l = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(l)

imagen_3c = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

cv2.imwrite("salida/clahe.png", imagen_3c)

for res in paddle_ocr.predict(imagen_3c):
    res.print()
    res.save_to_img("salida")        
    res.save_to_json("salida") 

In [ ]:
# ADAPTATIVE

gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
bw = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY, 31, 15)

imagen_3c = cv2.cvtColor(bw, cv2.COLOR_GRAY2BGR) 

cv2.imwrite("salida/adaptative.png", imagen_3c)

for res in paddle_ocr.predict(imagen_3c):
    res.print()
    res.save_to_img("salida")        
    res.save_to_json("salida") 


In [ ]:

def brillo_contraste(img, alpha=1.3, beta=20):
    # alpha: contraste (1.0 = igual), beta: brillo (-100..100)
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)


def gamma(img, g=0.6):
    # g < 1 aclara, g > 1 oscurece
    tabla = np.array([(i / 255.0) ** g * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(img, tabla)



def normalizar(img):
    return cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)


### Vamos a probar con otro OCR

In [ ]:
path_img = Path("pruebas14082026/4d95d836")
img = cv2.imread(str(path_img / "original.png"))

In [ ]:
if img is None:
    raise FileNotFoundError(
        f"No se encuentra la imagen: {path_img}"
    )

# Convertir a escala de grises
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Mejorar el contraste mediante binarización
processed = cv2.threshold(
    gray,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)[1]

# Ejecutar OCR
config = "--psm 6"
text = pytesseract.image_to_string(
    processed,
    config=config
)

# Mostrar el texto reconocido en el terminal
print("Texto reconocido:")
print(text)

# Mostrar la imagen original y la procesada
cv2.imshow("Imagen original", img)
cv2.imshow("Imagen procesada", processed)

cv2.waitKey(0)
cv2.destroyAllWindows()